# Agros · training on Google Colab

Three experiments, each tested on data the model never saw, with every score printed as measured:

1. **Harvest years** (Chakra agent's harvest answers): 10 machine-learning models and 2 deep-learning models (1D-CNN, GRU)
   predict whether a crop's national harvest is good, average or bad from that season's NASA POWER weather.
   Test: **leave one whole year out**, the same as the in-app model (`scripts/build/train_harvest_model.mjs`).
2. **Crop doctor images** (tomato): a MobileNetV3 network fine-tuned on PlantVillage (lab photos), then tested on
   **PlantDoc** (photos taken in real fields). Exported as SavedModel and TFLite for offline use.
3. **Chakra agent retrieval**: does a multilingual embedding model find the right knowledge-base passage more often than
   the app's BM25? Tested on 44 hand-written farmer questions.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (part 2 needs it; parts 1 and 3 run on CPU too).
Run the cells in order. Each part ends by saving a results JSON you can download.

**About accuracy targets.** A score is only worth something on data the model did not see. The notebook prints the
measured held-out scores even when they are low, and part 1 shows how a leaky split inflates them. Lab-photo accuracy
(PlantVillage) is usually far above field accuracy (PlantDoc). Report the field number.

Data: FAOSTAT and NASA POWER (real, `npm run colab:data` in the repo writes `notebooks/data/`). PlantVillage
(github.com/spMohanty/PlantVillage-Dataset; the repository states no license, so check before redistributing a model
trained on it). PlantDoc (github.com/pratikkayal/PlantDoc-Dataset, CC-BY-4.0).

In [ ]:
#@title 0 · Get the data (clone the repo, or upload notebooks/data/*)
import os, json, subprocess, pathlib, datetime
REPO = "https://github.com/razibr246/AgroVision.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
NEED = ['harvest_table.csv', 'chakra_chunks.json', 'chakra_retrieval_eval.json']
DATA = pathlib.Path('/content/agros_data')

def clone():
    if not pathlib.Path('/content/repo').exists():
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, '--filter=blob:none', '--sparse', REPO, '/content/repo'],
                           capture_output=True, text=True)
        if r.returncode:
            print('clone failed (private repo or no notebooks/data yet):', r.stderr[-300:]); return None
        subprocess.run(['git', '-C', '/content/repo', 'sparse-checkout', 'set', 'notebooks/data'], check=True)
    d = pathlib.Path('/content/repo/notebooks/data')
    return d if all((d / f).exists() for f in NEED) else None

found = clone()
if found:
    DATA = found
else:
    from google.colab import files
    DATA.mkdir(exist_ok=True)
    print('Upload these files from notebooks/data/ in your copy of the repo:', NEED)
    for name, b in files.upload().items():
        (DATA / name).write_bytes(b)
missing = [f for f in NEED if not (DATA / f).exists()]
assert not missing, f'missing {missing}: run `npm run colab:data` in the repo, then upload notebooks/data/*'
print('data in', DATA)

In [ ]:
#@title Install libraries (most are already in Colab)
!pip -q install xgboost lightgbm rank_bm25 sentence-transformers

## Part 1 · Good and bad harvest years

Each row is one crop's national harvest in one year (FAOSTAT). The label compares that year's yield with the crop's own
trend (the 7 years either side): **good** at +4% or more, **bad** at −4% or less, **average** between. The clues are the
season's weather averaged over the seven Agros upazilas.

Every model is tested the same way: for each year, train on all other years and predict that year's harvests of every
crop. A model never sees the year it predicts, so it can't copy that year's weather from another crop's record.

In [ ]:
#@title 1 · Load the harvest table
import numpy as np, pandas as pd
df = pd.read_csv(DATA / 'harvest_table.csv')
CLASSES = ['bad', 'average', 'good']
y = df['label'].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
years = df['year'].to_numpy()
FEAT = ['rain', 'heavyDays', 'dryRun', 'tmaxMean', 'hotDays', 'coldNights', 'rhMean', 'sradMean']
VARS = ['prec', 'tmax', 'tmin', 'rh', 'srad']
SEQ = [f'm{k}_{v}' for k in range(1, 13) for v in VARS]
season = pd.get_dummies(df['season'], prefix='season').astype(float)
crop = pd.get_dummies(df['crop'], prefix='crop').astype(float)
FEATURE_SETS = {
    'crop + season only (no weather)': pd.concat([season, crop], axis=1),  # the floor: what knowing the crop alone gives
    'weather + season (as in the app)': pd.concat([df[FEAT], season], axis=1),
    '+ crop + 12-month weather': pd.concat([df[FEAT], season, crop, df[SEQ]], axis=1),
}
print(f'{len(df)} crop-years, {df.crop.nunique()} crops, {df.year.min()}–{df.year.max()}, {df.year.nunique()} years')
print(df['label'].value_counts().to_string())

In [ ]:
#@title 2 · Ten machine-learning models, leave one whole year out
import warnings; warnings.filterwarnings('ignore')
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, recall_score, balanced_accuracy_score, confusion_matrix
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

SEED = 7
LOYO = LeaveOneGroupOut()

def score(yt, pred):
    return dict(accuracy=accuracy_score(yt, pred), macroF1=f1_score(yt, pred, average='macro'),
                balancedAcc=balanced_accuracy_score(yt, pred),
                badRecall=recall_score(yt, pred, labels=[0], average=None)[0])

def pipe(m):  # imputing and scaling are fitted inside each training fold only
    return make_pipeline(SimpleImputer(strategy='mean'), StandardScaler(), m)

# Settings are fixed in advance, not tuned on these scores (tuning on the test years would flatter them).
MODELS = {
    'base rate (always the commonest)': DummyClassifier(strategy='most_frequent'),
    'logistic regression': LogisticRegression(C=0.5, class_weight='balanced', max_iter=3000),
    'kNN (k=15)': KNeighborsClassifier(15),
    'SVM (RBF)': SVC(C=1.0, class_weight='balanced'),
    'random forest': RandomForestClassifier(500, min_samples_leaf=5, class_weight='balanced', random_state=SEED, n_jobs=-1),
    'extra trees': ExtraTreesClassifier(500, min_samples_leaf=5, class_weight='balanced', random_state=SEED, n_jobs=-1),
    'hist. gradient boosting': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, class_weight='balanced', random_state=SEED),
    'XGBoost': XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, eval_metric='mlogloss'),
    'LightGBM': LGBMClassifier(n_estimators=300, num_leaves=7, learning_rate=0.05, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, class_weight='balanced', random_state=SEED, verbose=-1),
    'MLP (small neural net)': MLPClassifier((32, 16), alpha=1e-2, max_iter=2000, early_stopping=True, random_state=SEED),
}
rows, PRED = [], {}
for fs, X in FEATURE_SETS.items():
    for name, m in MODELS.items():
        p = cross_val_predict(pipe(m), X.to_numpy(dtype=float), y, groups=years, cv=LOYO)
        PRED[(fs, name)] = p
        rows.append(dict(features=fs, model=name, **score(y, p)))
        r = rows[-1]
        print(f"{fs[:26]:26s} | {name:32s} acc {r['accuracy']:.3f}  macro-F1 {r['macroF1']:.3f}  bad years caught {r['badRecall']:.3f}")

In [ ]:
#@title 3 · Deep learning: 1D-CNN and GRU over the 12 months of weather (same test)
import tensorflow as tf
from tensorflow import keras
tf.keras.utils.set_random_seed(SEED)
seq = df[SEQ].to_numpy(dtype='float32').reshape(-1, 12, len(VARS))
static = pd.concat([season, crop], axis=1).to_numpy(dtype='float32')

def build(kind):
    s_in, t_in = keras.Input((12, len(VARS))), keras.Input((static.shape[1],))
    if kind == '1D-CNN':
        h = keras.layers.Conv1D(16, 3, padding='same', activation='relu')(s_in)
        h = keras.layers.Conv1D(16, 3, padding='same', activation='relu')(h)
        h = keras.layers.GlobalAveragePooling1D()(h)
    else:
        h = keras.layers.GRU(16)(s_in)
    h = keras.layers.Concatenate()([h, t_in])
    h = keras.layers.Dense(16, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-3))(h)
    out = keras.layers.Dense(3, activation='softmax')(keras.layers.Dropout(0.3)(h))
    m = keras.Model([s_in, t_in], out)
    m.compile(keras.optimizers.Adam(1e-3), 'sparse_categorical_crossentropy')
    return m

def loyo_dl(kind):
    pred = np.zeros(len(y), dtype=int)
    rng = np.random.default_rng(SEED)
    for tr, te in LOYO.split(seq, y, years):
        tr = rng.permutation(tr)  # so the validation slice is a random 15%, not the last crops
        mu, sd = np.nanmean(seq[tr], axis=0), np.nanstd(seq[tr], axis=0) + 1e-6
        norm = lambda a: np.nan_to_num((a - mu) / sd)  # missing months (before 2001) become the training mean
        cw = len(tr) / (3 * np.bincount(y[tr], minlength=3).clip(1))
        m = build(kind)
        m.fit([norm(seq[tr]), static[tr]], y[tr], epochs=80, batch_size=32, verbose=0, validation_split=0.15,
              class_weight=dict(enumerate(cw)), callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)])
        pred[te] = m.predict([norm(seq[te]), static[te]], verbose=0).argmax(1)
    return pred

for kind in ['1D-CNN', 'GRU']:
    p = loyo_dl(kind)
    fs = 'DL: 12-month weather + season + crop'
    PRED[(fs, kind)] = p
    rows.append(dict(features=fs, model=kind, **score(y, p)))
    r = rows[-1]
    print(f"{kind:8s} acc {r['accuracy']:.3f}  macro-F1 {r['macroF1']:.3f}  bad years caught {r['badRecall']:.3f}")

In [ ]:
#@title 4 · Results, and two honesty checks
res = pd.DataFrame(rows).sort_values('macroF1', ascending=False).reset_index(drop=True)
display(res.round(3))
best = res.iloc[0]
print(f"\nBest by macro-F1: {best.model} ({best.features}) - accuracy {best.accuracy:.1%}, bad years caught {best.badRecall:.1%}")
base = res[res.model.str.startswith('base rate')].accuracy.max()
print(f"Always saying 'average' is right {base:.1%} of the time, so accuracy alone flatters; macro-F1 and bad-year recall are the fair scores.")
print('Confusion matrix of the best (rows = truth bad/average/good):')
print(confusion_matrix(y, PRED[(best.features, best.model)]))

# (a) Luck: shuffle the labels within each crop and score the same model again, five times.
X_best = FEATURE_SETS.get(best.features, FEATURE_SETS['+ crop + 12-month weather']).to_numpy(dtype=float)
m_best = MODELS.get(best.model, MODELS['logistic regression'])
rng = np.random.default_rng(SEED); luck = []
for _ in range(5):
    ys = y.copy()
    for c in df.crop.unique():
        ix = np.where(df.crop == c)[0]; ys[ix] = rng.permutation(ys[ix])
    luck.append(score(ys, cross_val_predict(pipe(m_best), X_best, ys, groups=years, cv=LOYO))['macroF1'])
print(f"\n(a) Same model on labels shuffled within each crop: macro-F1 {np.mean(luck):.3f} ± {np.std(luck):.3f}. That is what crop base rates plus luck give; the real score must clear it.")

# (b) A leaky split (random 5-fold, years mixed between train and test). Shown only to see the inflation. Do not report it.
leak = score(y, cross_val_predict(pipe(m_best), X_best, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED)))
gap = leak['accuracy'] - best.accuracy
print(f"(b) Leaky random split: accuracy {leak['accuracy']:.1%} vs {best.accuracy:.1%} held out by year"
      + (f" - {gap:.1%} higher, which is how much a wrong test flatters." if gap > 0.005 else " - no inflation here, but only the by-year test is safe to report."))
floor = res[res.features.str.startswith('crop + season only')].macroF1.max()
print(f"(c) Crop identity alone reaches macro-F1 {floor:.3f}; the best model {best.macroF1:.3f}. The weather adds {best.macroF1 - floor:+.3f}.")
print(f"    Picking the best of {len(res)} runs on the same test also flatters it a little; treat a lead of under ~0.02 as a tie.")
print(f"\nTarget asked for: 97%. Measured held-out accuracy: {best.accuracy:.1%}. National yields swing on seed, fertiliser, pests, prices and policy too, which the weather does not show.")

In [ ]:
#@title 5 · Save the harvest results
harvest_out = dict(
    status='REAL', generated=datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
    data=dict(rows=int(len(df)), crops=int(df.crop.nunique()), years=[int(df.year.min()), int(df.year.max())],
              source='FAOSTAT Production_Crops_Livestock (Bangladesh) + NASA POWER, notebooks/data/harvest_table.csv'),
    method='Leave one whole year out; settings fixed in advance; scaling and imputing fitted on each training fold.',
    results=res.round(4).to_dict('records'), best=dict(model=best.model, features=best.features),
    luck_macroF1=dict(mean=float(np.mean(luck)), sd=float(np.std(luck))), crop_only_macroF1=float(floor), leaky_split_not_for_report=leak,
)
json.dump(harvest_out, open('/content/harvest_colab_results.json', 'w'), indent=1, default=float)
print('saved /content/harvest_colab_results.json')

## Part 2 · Crop doctor: tomato leaf photos

Training data: **PlantVillage** tomato (10 classes, about 18,000 photos of single leaves on a plain background).
Field test: **PlantDoc** tomato (photos taken in real fields; 8 of its classes match PlantVillage's). A model that scores 99% on
PlantVillage can fall far lower on PlantDoc, and only the field number tells you how it will do on a farmer's photo.

Run A trains on PlantVillage only. Run B also adds PlantDoc's *training* photos, the honest way to raise field accuracy,
and is still tested on PlantDoc's separate *test* photos.

In [ ]:
#@title 6 · Download PlantVillage tomato and PlantDoc tomato (sparse git clone, a few minutes)
PV, PD = pathlib.Path('/content/plantvillage'), pathlib.Path('/content/plantdoc')
TOMATO_PV = ['Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold',
             'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot',
             'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy']
# PlantDoc folder -> PlantVillage class. Spider mites and target spot have no PlantDoc folder.
PD_TO_PV = {'Tomato leaf bacterial spot': 'Tomato___Bacterial_spot', 'Tomato Early blight leaf': 'Tomato___Early_blight',
            'Tomato leaf late blight': 'Tomato___Late_blight', 'Tomato mold leaf': 'Tomato___Leaf_Mold',
            'Tomato Septoria leaf spot': 'Tomato___Septoria_leaf_spot', 'Tomato leaf yellow virus': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
            'Tomato leaf mosaic virus': 'Tomato___Tomato_mosaic_virus', 'Tomato leaf': 'Tomato___healthy'}
# The app's own problem keys (src/data/crop_problems.js) for the classes that have one.
APP_KEY = {'Tomato___Early_blight': 'early_blight', 'Tomato___Late_blight': 'late_blight', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus': 'leaf_curl'}

def sparse(url, dest, patterns):
    if dest.exists(): return
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', url, str(dest)], check=True)
    subprocess.run(['git', '-C', str(dest), 'sparse-checkout', 'set', '--no-cone', *patterns], check=True)

sparse('https://github.com/spMohanty/PlantVillage-Dataset.git', PV, [f'/raw/color/{c}/' for c in TOMATO_PV])
sparse('https://github.com/pratikkayal/PlantDoc-Dataset.git', PD, ['/train/Tomato*/', '/test/Tomato*/'])

CLASS_IDX = {c: i for i, c in enumerate(TOMATO_PV)}
def files_of(root, mapping):
    out = []
    for d in sorted(root.iterdir()):
        if d.is_dir() and d.name in mapping:
            out += [(str(f), CLASS_IDX[mapping[d.name]]) for f in sorted(d.iterdir()) if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    return out
pv_all = files_of(PV / 'raw/color', {c: c for c in TOMATO_PV})
pd_train, pd_test = files_of(PD / 'train', PD_TO_PV), files_of(PD / 'test', PD_TO_PV)
for name, f in [('PlantVillage', pv_all), ('PlantDoc train', pd_train), ('PlantDoc test', pd_test)]:
    print(f'{name:15s} {len(f):6d} photos', np.bincount([l for _, l in f], minlength=10))
assert pv_all and pd_test, 'download failed: check the two repositories are reachable'

In [ ]:
#@title 7 · Build the image pipelines and the model (MobileNetV3, small enough for a phone)
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers
IMG, BATCH = 224, 32
labels_of = lambda items: [l for _, l in items]
# PlantVillage holds several photos of the same leaf, so a random split leaks a little; PlantDoc is the clean test.
pv_tr, pv_rest = train_test_split(pv_all, test_size=0.2, stratify=labels_of(pv_all), random_state=SEED)
pv_va, pv_te = train_test_split(pv_rest, test_size=0.5, stratify=labels_of(pv_rest), random_state=SEED)

def load(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.image.resize(img, (IMG, IMG)), label
def make_ds(items, train=False):
    d = tf.data.Dataset.from_tensor_slices(([a for a, _ in items], labels_of(items)))
    if train: d = d.shuffle(len(items), seed=SEED, reshuffle_each_iteration=True)
    return d.map(load, num_parallel_calls=tf.data.AUTOTUNE).ignore_errors().batch(BATCH).prefetch(tf.data.AUTOTUNE)

# Field-style augmentation: angles, zoom, uneven light.
augment = keras.Sequential([layers.RandomFlip('horizontal_and_vertical'), layers.RandomRotation(0.15), layers.RandomZoom(0.25),
                            layers.RandomContrast(0.3), layers.RandomBrightness(0.3, value_range=(0, 255))])
def make_model():
    base = keras.applications.MobileNetV3Large(input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet',
                                               include_preprocessing=True, pooling='avg')
    base.trainable = False
    inp = keras.Input((IMG, IMG, 3))
    out = layers.Dense(len(TOMATO_PV), activation='softmax')(layers.Dropout(0.3)(base(augment(inp), training=False)))
    return keras.Model(inp, out), base

def train(train_items, val_items, epochs_head=6, epochs_fine=6):
    model, base = make_model()
    model.compile(keras.optimizers.Adam(1e-3), 'sparse_categorical_crossentropy', metrics=['accuracy'])
    tr, va = make_ds(train_items, True), make_ds(val_items)
    model.fit(tr, validation_data=va, epochs=epochs_head)
    base.trainable = True   # then fine-tune the top of the network, keeping batch-norm statistics fixed
    for l in base.layers[:-60]: l.trainable = False
    for l in base.layers:
        if isinstance(l, layers.BatchNormalization): l.trainable = False
    model.compile(keras.optimizers.Adam(1e-5), 'sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(tr, validation_data=va, epochs=epochs_fine,
              callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True, monitor='val_accuracy')])
    return model

def wilson(k, n, z=1.96):
    if not n: return (0, 0)
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)
def evaluate(model, items, name):
    truth, pred = [], []
    for xb, yb in make_ds(items):  # labels come with their own photos, so a skipped unreadable file cannot shift them
        pred.append(model.predict_on_batch(xb).argmax(1)); truth.append(yb.numpy())
    truth, pred = np.concatenate(truth), np.concatenate(pred)
    k = int((pred == truth).sum()); lo, hi = wilson(k, len(truth))
    print(f'{name:32s} accuracy {k / len(truth):.1%}  (95% CI {lo:.1%}–{hi:.1%}, {len(truth)} photos)')
    return dict(accuracy=k / len(truth), ci95=[lo, hi], n=int(len(truth)),
                confusion=confusion_matrix(truth, pred, labels=range(len(TOMATO_PV))).tolist())

In [ ]:
#@title 8 · Run A (PlantVillage only) and run B (+ PlantDoc training photos), both tested on PlantDoc test
model_a = train(pv_tr, pv_va)
A = dict(lab=evaluate(model_a, pv_te, 'A · PlantVillage test (lab)'), field=evaluate(model_a, pd_test, 'A · PlantDoc test (field)'))

# Run B: field photos are few, so each is shown 5 times as often as a lab photo. Validation stays on lab photos,
# so the PlantDoc test photos are never used to pick a model.
model_b = train(pv_tr + pd_train * 5, pv_va)
B = dict(lab=evaluate(model_b, pv_te, 'B · PlantVillage test (lab)'), field=evaluate(model_b, pd_test, 'B · PlantDoc test (field)'))
best_name, best_img = ('B', model_b) if B['field']['accuracy'] >= A['field']['accuracy'] else ('A', model_a)
print(f"\nUse run {best_name}. Field accuracy is the number to report; lab accuracy only shows the network learned the lab photos.")

In [ ]:
#@title 9 · Export the doctor model (SavedModel + TFLite) with its labels and scores
best_img.export('/content/agros_tomato_doctor')
conv = tf.lite.TFLiteConverter.from_saved_model('/content/agros_tomato_doctor')
conv.optimizations = [tf.lite.Optimize.DEFAULT]
open('/content/agros_tomato_doctor.tflite', 'wb').write(conv.convert())
doctor_out = dict(
    status='MODEL', generated=datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z', crop='tomato', run=best_name,
    input=dict(size=[IMG, IMG, 3], range='0-255 RGB (preprocessing is inside the model)'),
    classes=[dict(id=c, appKey=APP_KEY.get(c)) for c in TOMATO_PV],
    scores=dict(A=A, B=B), train=dict(plantvillage=len(pv_tr), plantdoc=len(pd_train) if best_name == 'B' else 0),
    note='Field accuracy is measured on PlantDoc test photos. A first look only: an officer confirms every diagnosis.',
)
json.dump(doctor_out, open('/content/doctor_colab_results.json', 'w'), indent=1, default=float)
print('saved /content/agros_tomato_doctor (SavedModel), /content/agros_tomato_doctor.tflite, /content/doctor_colab_results.json')

In [ ]:
#@title (optional) TensorFlow.js copy for running in the browser
# tensorflowjs can clash with Colab's preinstalled packages; if this fails, the TFLite file above still works.
!pip -q install tensorflowjs
!tensorflowjs_converter --input_format=tf_saved_model /content/agros_tomato_doctor /content/agros_tomato_doctor_tfjs && ls -la /content/agros_tomato_doctor_tfjs

## Part 3 · Chakra agent: finding the right passage

The agent answers from its knowledge base (149 passages on fields, crops, diseases, districts, support programmes and
climate). Before an AI model writes anything, the app has to *find* the right passage. The app uses BM25 (word
matching); in the app it finds the right passage first for 35 of these 44 questions and in the top three for 39.
Here three multilingual embedding models, and BM25 + embedding combined, are tested on the same 44 questions.
With 44 questions one question is about 2 points, so small differences are noise (see the intervals).

In [ ]:
#@title 10 · BM25 against multilingual embeddings
import re
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
chunks = json.load(open(DATA / 'chakra_chunks.json'))['chunks']
qs = json.load(open(DATA / 'chakra_retrieval_eval.json'))['questions']
docs = [f"{c['title']}\n{c.get('bn') or ''}\n{c.get('en') or ''}" for c in chunks]
target = []
for q in qs:
    hit = [i for i, c in enumerate(chunks) if q['expect'] in c['title']]
    assert len(hit) == 1, (q['expect'], len(hit)); target.append(hit[0])
target = np.array(target)

tok = lambda s: [t for t in re.split(r"[\s,.;:!?()\[\]\"'“”‘’/।\-–]+", s.lower()) if t]
bm25 = BM25Okapi([tok(d) for d in docs])
def rank_bm25(q): return np.argsort(-bm25.get_scores(tok(q)))

def metrics(ranks):
    pos = np.array([int(np.where(r == t)[0][0]) for r, t in zip(ranks, target)])
    h1 = pos == 0; boot = np.random.default_rng(SEED).choice(h1, (2000, len(h1))).mean(1)
    return dict(hit1=float(h1.mean()), hit3=float((pos < 3).mean()), mrr10=float(np.where(pos < 10, 1 / (pos + 1), 0).mean()),
                hit1_ci95=[float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))])

RET = {'BM25 (python, close to the app)': metrics([rank_bm25(q['q']) for q in qs])}
EMB = {  # (model, query prefix, passage prefix)
    'multilingual-MiniLM-L12 (small, browser-sized)': ('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', '', ''),
    'multilingual-e5-base': ('intfloat/multilingual-e5-base', 'query: ', 'passage: '),
    'BGE-M3': ('BAAI/bge-m3', '', ''),
}
dense_rank = {}
for name, (mid, qp, pp) in EMB.items():
    m = SentenceTransformer(mid)
    D = m.encode([pp + d for d in docs], normalize_embeddings=True, batch_size=16, show_progress_bar=False)
    Q = m.encode([qp + q['q'] for q in qs], normalize_embeddings=True)
    dense_rank[name] = [np.argsort(-(D @ v)) for v in Q]
    RET[name] = metrics(dense_rank[name])
    # Hybrid: reciprocal-rank fusion of BM25 and this embedding (k = 60, the usual constant).
    fused = []
    for qi, q in enumerate(qs):
        s = np.zeros(len(docs))
        for r in (rank_bm25(q['q']), dense_rank[name][qi]):
            s[r] += 1 / (60 + np.arange(1, len(docs) + 1))
        fused.append(np.argsort(-s))
    RET[f'BM25 + {name}'] = metrics(fused)

ret = pd.DataFrame(RET).T.sort_values('hit1', ascending=False)
display(ret[['hit1', 'hit3', 'mrr10']].astype(float).round(3).assign(hit1_ci95=ret.hit1_ci95.map(lambda c: f'{c[0]:.0%}–{c[1]:.0%}')))

In [ ]:
#@title 11 · Save the retrieval results, and download everything
retrieval_out = dict(status='MODEL', generated=datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
                     questions=len(qs), passages=len(chunks), app_bm25=dict(hit1=35 / 44, hit3=39 / 44, note='measured in the app before this run'),
                     results=RET, note='44 hand-written questions (scripts/build/chakra_retrieval_eval.json); one question is about 2 points.')
json.dump(retrieval_out, open('/content/chakra_retrieval_colab_results.json', 'w'), indent=1)
from google.colab import files
for f in ['harvest_colab_results.json', 'doctor_colab_results.json', 'chakra_retrieval_colab_results.json', 'agros_tomato_doctor.tflite']:
    if pathlib.Path('/content/' + f).exists(): files.download('/content/' + f)

## After the run

Put the three `*_colab_results.json` files in `src/data/models/colab/` in the repo, and the `.tflite` (or the TF.js
folder) in `public/models/doctor/`. Keep each score as measured. Using a model in the app is a separate change:
the harvest model's scores go on `/science#harvest`; a tomato image model can give the crop doctor an offline first
look beside the existing `engine/cropDoctor.js` (label it `MODEL`, and keep the officer's confirmation); an embedding
retriever only belongs in the Chakra agent if it beats BM25 by more than the interval above.